In [1]:
import numpy as np

def longrad(opt, Ld, TW, TWe, TG, TGe, TR, eW, eWe, eG, eGe, eR, FGS, FWW, FGW, FWG, FWS, nW, nG, nR):
    # Stephan-Boltzmann constant
    ss = 5.67e-8  # [J/s/m^2/K^4]
    FG = FGS
    FW = FWS

    # Initialize variables
    LG1 = np.zeros(nG)
    LG2 = np.zeros(nG)
    LW1 = np.zeros(nW)
    LW2 = np.zeros(nW)
    Lr = np.zeros(nR)

    if opt == 1:
        # Compute longwave radiation for option 1
        for j1 in range(nR):
            Lr[j1] = eR[j1] * (Ld - ss * TR[j1] ** 4)

        for j2 in range(nW):
            LW1[j2] = eW[j2] * (Ld * FWS + eGe * ss * TGe ** 4 * FWG +
                                eW[j2] * ss * TW[j2] ** 4 * FWW - ss * TW[j2] ** 4)
            LW2[j2] = eW[j2] * ((1 - eGe) * Ld * FGS * FWG +
                                2 * (1 - eGe) * eW[j2] * ss * TW[j2] ** 4 * FGW * FWG +
                                (1 - eW[j2]) * Ld * FWS * FWW +
                                (1 - eW[j2]) * eGe * ss * TGe ** 4 * FWG * FWW +
                                eW[j2] * (1 - eW[j2]) * ss * TW[j2] ** 4 * FWW * FWW)

        for j3 in range(nG):
            LG1[j3] = eG[j3] * (Ld * FGS + 2 * eWe * ss * TWe ** 4 * FGW - ss * TG[j3] ** 4)
            LG2[j3] = 2 * eG[j3] * ((1 - eWe) * Ld * FWS * FGW +
                                     (1 - eWe) * eG[j3] * ss * TG[j3] ** 4 * FWG * FGW +
                                     eWe * (1 - eWe) * ss * TWe ** 4 * FWW * FGW)

        Lw = LW1 + LW2
        Lg = LG1 + LG2

    elif opt == 2:
        # Compute longwave radiation for option 2
        tem = 0
        Lg = np.zeros(nG)

        for j in range(nG):
            Lg[j] = eG[j] * FG * Ld - eG[j] * ss * TG[j] ** 4 + \
                    eG[j] * eW * (1 - FG) * ss * TW[0] ** 4 + \
                    eG[j] * (1 - eW) * (1 - FG) * FW * Ld + \
                    eG[j] * eW * (1 - eW) * (1 - FG) * (1 - 2 * FW) * ss * TW[0] ** 4 + \
                    eG[j] * (1 - eW) * (1 - FG) * FW * ss * eG[j] * TG[j] ** 4

            tem += eW * FW * ss * eG[j] * TG[j] ** 4 + \
                   eW * (1 - eG[j]) * FW * FG * Ld + \
                   eW ** 2 * (1 - eG[j]) * FW * (1 - FG) * ss * TW[0] ** 4 + \
                   eW * (1 - eW) * FW * (1 - 2 * FW) * ss * eG[j] * TG[j] ** 4

        Lw = tem + eW * FW * Ld - eW * ss * TW[0] ** 4 + \
             eW ** 2 * (1 - 2 * FW) * ss * TW[0] ** 4 + \
             eW * (1 - eW) * FW * (1 - 2 * FW) * Ld + \
             eW ** 2 * (1 - eW) * (1 - 2 * FW) ** 2 * ss * TW[0] ** 4

        Lr = eR * (Ld - ss * TR[0] ** 4)

    return Lw, Lg, Lr
